In [ ]:
source("../paths.R")   # SCREEN_DATA, REFERENCE, TEXTFILES; see data/README.md

# NOTE: Main.R, Conf.R, Utilities.R and DAVID_query.R are deliberately not
# sourced. Conf.R pointed at a machine that no longer exists, the DAVID call
# below is commented out, and the two helpers this notebook uses are inlined.
# ICtest is not loaded either: nothing here calls it, and once attached it
# breaks every ggplot "+" chain for the rest of the session.
library(ica)            # icaimax()
library(pheatmap)
library(corrplot)
library(RColorBrewer)
library(factoextra)     # fviz_eig(), the scree plots
library(repr)           # options(repr.plot.*)

# maptree, NMF and ggpubr were loaded by the original and are not used here.

lappend <- function(lst, obj) { lst[[length(lst) + 1]] <- obj; lst }

# Writes a ragged list of character vectors as one column each, padded with
# blanks. This is what produces the factor table.
writePathwayFile <- function(pathwayGenes, fileName, cNames = FALSE) {
    maxlen <- max(lengths(pathwayGenes))
    padded <- lapply(pathwayGenes, function(x) c(x, rep(NA, maxlen - length(x))))
    df <- do.call("cbind", lapply(padded, as.data.frame))
    colnames(df) <- names(pathwayGenes)
    write.table(df, fileName, sep = ",", row.names = FALSE, na = " ",
                quote = FALSE, col.names = cNames)
}

geneCutOff   <- 4
targetCutOff <- 15
nFactors     <- 15

In [ ]:
writePathwayFile <- function(pathwayGenes, fileName, cNames=F){
  maxlen <- max(lengths(pathwayGenes))
  pathwayGenes2 <- lapply(pathwayGenes, function(lst) c(lst, rep(NA, maxlen - length(lst))))

  pathwayGenes.df <- do.call("cbind", lapply(pathwayGenes2, as.data.frame)) 
  colnames(pathwayGenes.df) <- names(pathwayGenes)

  write.table(pathwayGenes.df,  fileName, sep=",", row.names = F, na = " ", quote = F, col.names = cNames)
}

# dcGenes <- read.csv("/home/beraslan/jovian-work/analysisSingle/PositiveControls/DC_genes.csv", row.names = 1, stringsAsFactors=FALSE)
                          
# k = dcGenes[1,"Genes"]
# length(strsplit(k, ",")[[1]])
                          
# dcList = list()
# for(i in 1:nrow(dcGenes)){
#     dcList <- lappend(dcList, strsplit(dcGenes[i,"Genes"],",")[[1]])
# }
# names(dcList) <- rownames(dcGenes)
                          
# writePathwayFile(pathwayGenes=dcList, fileName="/home/beraslan/jovian-work/analysisSingle/PositiveControls/DC_cellstate_genes.csv", cNames=names(dcList))                          

In [ ]:
selCoefs <- read.csv(referencePath("SignificantCoefMatrix.csv"), row.names = 1)
selCoefs <- t(selCoefs)
KOGenes <-  colnames(selCoefs)
KOGenes <- sapply(KOGenes, function(x){strsplit(x,"_")[[1]][2]})
colnames(selCoefs) <- KOGenes

In [ ]:
dim(selCoefs)

In [ ]:
boxplot(selCoefs)
abline(h=0.5, col="red")
abline(h= -0.5, col="red")

In [ ]:
#guideModulesN <- data.frame(read.csv("/home/beraslan/jovian-work/analysisSingle/ME_GuideModules_7_Modules.csv", row.names = 1))
#guideModulesN$GuideName <- sapply(guideModulesN$GuideName, function(x){strsplit(as.character(x),"_")[[1]][2]})

guideModulesN <- data.frame(read.csv(referencePath("ME_GuideModules_leiden_6_Modules.csv")),  row.names = 1)
rownames(guideModulesN) <- guideModulesN$GuideName 
guideModulesN$GuideName <- NULL
guideModulesN$GuideGroup <- factor(guideModulesN$GuideGroup)

head(guideModulesN)

In [ ]:
#geneModulesN <- data.frame(read.csv("/home/beraslan/jovian-work/analysisSingle/ME_GeneModules_6_Modules.csv", row.names = 1))
geneModulesN <- data.frame(read.csv(referencePath("ME_GeneModules_leiden_8_Modules.csv")), row.names = 1)
rownames(geneModulesN) <- geneModulesN$GeneName
geneModulesN$GeneName <- NULL
geneModulesN$GeneGroup <- factor(geneModulesN$GeneGroup)

head(geneModulesN)

In [ ]:
head(selCoefs)

In [ ]:
# k <- prcomp(selCoefs, rank. = 100)
# summary(k)

In [ ]:
dim(selCoefs)

In [ ]:
#FOBIasymp(X = as.matrix(selCoefs), k = 4, model = "ICA", type = "S1")


In [ ]:
#res0 <- NGPPsim(selCoefs, k = 20, N = 400)


In [ ]:
#FOBIboot(selCoefs, k = 10, s.boot = "B1")

In [ ]:
k <- icaimax(selCoefs, nc = 15, center = T)

In [ ]:
dim(k$S)

In [ ]:
dim(k$M)

In [ ]:
myH <- t(k$S)
myW <- k$M

In [ ]:
dim(myH)

In [ ]:
dim(myW)

In [ ]:
rownames(myH) <- 1:nrow(myH)
colnames(myW) <- 1:ncol(myW)


In [ ]:
rownames(myW) <- KOGenes
colnames(myH) <- rownames(selCoefs)

In [ ]:
head(myW)

In [ ]:
head(myH)

In [ ]:
head(selCoefs)

In [ ]:
M_targets = cor(t(myH), method = "pearson")
M_genes = cor((myW), method = "pearson")

In [ ]:
boxplot(M_targets, main="M targets")
abline(h=0.2, col="red")
abline(h= -0.2, col="red")
axis(2, at = seq(-0.5,0.5,0.1))


In [ ]:
boxplot(M_genes, main="M genes")
abline(h=0.2, col="red")
abline(h= -0.2, col="red")
axis(2, at = seq(-0.5,0.5,0.1))

In [ ]:
M_targets[M_targets > 0.2] = 0.2
M_targets[M_targets < -0.2] = -0.2

M_genes[M_genes > 0.3] = 0.3
M_genes[M_genes < -0.3] = -0.3

In [ ]:
# diag(M_targets) <- NA
# diag(M_genes) <- NA

In [ ]:
#fviz_nbclust(M_targets, kmeans, method = "wss", k.max = 19)+ggtitle("Optimal number of clusters in of factor covariance based on target loadings")


In [ ]:
#fviz_nbclust(M_genes, kmeans, method = "wss", k.max = 19)+ggtitle("Optimal number of clusters in of factor covariance based on gene loadings")


In [ ]:
options(repr.plot.width=9, repr.plot.height=8)

col3 = colorRampPalette(c("navy", "white", "red"))(50)
# corrplot 0.95 masks the lower triangle before it computes the AOE
# ordering, and eigen() then fails on the NAs. Ordering first and drawing
# the reordered matrix gives the same picture.
ord <- corrplot::corrMatOrder(M_genes, order = 'AOE')
corrplot::corrplot(M_genes[ord, ord], order = 'original', cl.lim=c(-0.3,0.3), col=col3, method = 'ellipse', is.corr = FALSE, tl.cex = 1.5, diag = FALSE, type = 'upper')

In [ ]:
# options(repr.plot.width=10, repr.plot.height=6)

# ctree=9
# plot(hmTarget$tree_row)
# abline(h=ctree, col="red", lty=2, lwd=2)


In [ ]:
options(repr.plot.width=9, repr.plot.height=8)

hmGene <- pheatmap(M_genes, clustering_method="ward.D2", show_colnames = TRUE, show_rownames = TRUE, 
                clustering_distance_rows="euclidean", clustering_distance_cols="euclidean",
                   main="Covariance matrix of factors based on effected gene loadings", colorRampPalette(c("navy", "white", "red"))(50))


In [ ]:
hmGene$tree_col$order


In [ ]:
options(repr.plot.width=10, repr.plot.height=6)
plot(hmGene$tree_row)


In [ ]:
res.pca <- prcomp(M_targets, scale = FALSE)
summary(res.pca)
fviz_eig(res.pca, choice = 'variance', ncp = 19)+ggtitle("Scree plot of factor covariance based on target loadings")



In [ ]:
res.pca <- prcomp(M_genes, scale = FALSE)
fviz_eig(res.pca, ncp = 19)+ ggtitle("Scree plot of factor covariance based on effected gene loadings")


In [ ]:
head(selCoefs)

In [ ]:
boxplot(selCoefs)

In [ ]:
geneModulesN$GeneGroup <- paste0("GeneGroup_", geneModulesN$GeneGroup)

In [ ]:
unique(geneModulesN)

In [ ]:
guideModulesN$GuideGroup <- paste0("KOGroup_", guideModulesN$GuideGroup)

In [ ]:
unique(guideModulesN)

In [ ]:
guideModulesN$GuideColor <- NULL
geneModulesN$GeneColor <- NULL

In [ ]:
plotFactorHeatmap <- function(coefsAll, sKOGenes, sEffectGenes, elem){
  tempDF <- t(coefsAll[sEffectGenes, sKOGenes])
  #tmpPVals <- pValsAllAdj[sTargets, sGenes]
  #tempDF[tmpPVals > 0.1] <- 0
  tempDF[tempDF > 0.5] = 0.5
  tempDF[tempDF < -0.5] = -0.5
   
  annoCols<-list(GeneGroup=c(GeneGroup_0="#1B9E77", GeneGroup_1="#D95F02", GeneGroup_2="#7570B3", GeneGroup_3="#E7298A", 
                        GeneGroup_4="#66A61E", GeneGroup_5="#E6AB02", GeneGroup_6="#A6761D", GeneGroup_7="#666666"),
                 GuideGroup=c(KOGroup_0="#1f77b4", KOGroup_1="#ff7f0e", KOGroup_2="#279e68", KOGroup_3="#d62728", 
                        KOGroup_4="#aa40fc", KOGroup_5="#8c564b"))
    
    
  annotDFcol <- subset(geneModulesN, rownames(geneModulesN) %in% colnames(tempDF))
  annotDFrow <- subset(guideModulesN, rownames(guideModulesN) %in% rownames(tempDF))
    
  options(repr.plot.width=16, repr.plot.height=9)
  km2 <- pheatmap(tempDF, main = paste0("Factor ",elem),
           na_col = "grey",  
           annotation_col = annotDFcol,
                  annotation_row = annotDFrow,
            annotation_colors = annoCols,
           clustering_method="ward.D2",
           clustering_distance_rows="euclidean",
           clustering_distance_cols="euclidean", colorRampPalette(c("navy", "white", "red"))(50))
  
  geneOrder = colnames(tempDF)[km2$tree_col$order]
  koOrder = rownames(tempDF)[km2$tree_row$order]
    
  geneClusters=cutree(km2$tree_col, h=1)
  i=1
  while(length(unique(geneClusters)) > 2){
    i = (i+0.5)
    geneClusters <- cutree(km2$tree_col, h=i)
  }
  
    
  geneClusters_one <- names(geneClusters[geneClusters==1])
  geneClusters_two <- names(geneClusters[geneClusters==2])
    
    
  guideClusters=cutree(km2$tree_row, h=1)
  i=1
  while(length(unique(guideClusters)) > 2){
    i = (i+0.5)
    guideClusters <- cutree(km2$tree_row, h=i)
  }
  
  guideClusters_one <- names(guideClusters[guideClusters==1])
  guideClusters_two <- names(guideClusters[guideClusters==2])
    
  return(list("geneClusters_one"=geneClusters_one, "geneClusters_two"=geneClusters_two,
              "guideClusters_one"=guideClusters_one, "guideClusters_two"=guideClusters_two,
             "geneOrder" = geneOrder, "koOrder" = koOrder))
}


#ensemblBackground <- mapIds(org.Mm.eg.db, keys = colnames(coefsAll), keytype = "SYMBOL", column= "ENSEMBL")


In [ ]:
getOutliers <- function(x){
  qX <- quantile(x)
  IQR = qX[4] - qX[2]
  minLev = qX[2] - IQR
  maxLev = qX[4] + IQR
  
  return(x[ x < minLev | x > maxLev])
}


In [ ]:
getOutliers2 <- function(x){
  qX <- quantile(x)
  IQR = qX[4] - qX[2]
  minLev = qX[2] - 1.3*IQR
  maxLev = qX[4] + 1.3*IQR
  
  return(x[ x < minLev | x > maxLev])
}


In [ ]:
pathwayGenes <- list()
pathwayKOs <- list()
pathwayEffectedGenes <- list()
pathwayEffectedUpGenes <- list()
pathwayEffectedDownGenes <- list()
factorGeneGroups <- list()
geneKOOrder <- list()


for(elem in c(1:nFactors)){
  print(paste0("############## FACTOR : ",elem, " ##################"))
  
  factor0X <- myW[,elem]
  names(factor0X) <- rownames(myW)
  sKOGenes <- names(getOutliers(factor0X))
  print(length(sKOGenes))
    
  factor0Y <- myH[elem,]
  names(factor0Y) <- colnames(myH)
  sEffectGenes <- names(getOutliers2(factor0Y))
  print(length(sEffectGenes))
 
  boxplot(selCoefs)
  options(repr.plot.width=16, repr.plot.height=9)

  k <- plotFactorHeatmap(selCoefs, sKOGenes, sEffectGenes, elem)

  names(k) <- c(paste0("Factor ", elem, " gene group 1"), paste0("Factor ", elem, " gene group 2"),
                paste0("Factor ", elem, " guide group 1"),paste0("Factor ", elem, " guide group 2"),
                paste0("Factor ", elem, " gene order"), paste0("Factor ", elem, " guide order") )
  factorGeneGroups <- lappend(factorGeneGroups, k[1:4])
  geneKOOrder <- lappend(geneKOOrder, k[5:6])

  print("Genes in the pathway: ")
  pGenes = c(sEffectGenes, sKOGenes)

  pathwayGenes <- lappend(pathwayGenes, pGenes)
  pathwayKOs <- lappend(pathwayKOs, sKOGenes)
  pathwayEffectedGenes <- lappend(pathwayEffectedGenes, sEffectGenes)
  
  #getDAVIDGOTerms (pGenes, ensemblBackground, fNamePrefix=paste0("NMF_",inT, "_", elem), titleStr=paste0("Factor ",elem,))

}


In [ ]:
length(unique(unlist(pathwayKOs)))

In [ ]:
sort(unique(unlist(pathwayKOs)))

In [ ]:

length(unique(unlist(pathwayEffectedGenes)))

In [ ]:
saveRDS(geneKOOrder, "outputs/factorGeneGroups.rds")

In [ ]:
KOFactorMatrix = data.frame(matrix(0, ncol = length(unique(unlist(pathwayKOs))), nrow= nFactors))
rownames(KOFactorMatrix) <- paste0("Factor ", 1:nrow(KOFactorMatrix))
colnames(KOFactorMatrix) <- sort(unique(unlist(pathwayKOs)))

In [ ]:
for(i in 1:length(pathwayKOs)){
    facKOGenes = pathwayKOs[[i]]
    KOFactorMatrix[i,facKOGenes] = 1
}

In [ ]:
head(KOFactorMatrix)

In [ ]:
options(repr.plot.width=26, repr.plot.height=6)

pheatmap(KOFactorMatrix, color = c("white","blue"), cluster_rows = F, main = "KO genes per factor")

In [ ]:
EGenesFactorMatrix = data.frame(matrix(0, ncol = length(unique(unlist(pathwayEffectedGenes))), nrow= nFactors))
rownames(EGenesFactorMatrix) <- paste0("Factor ", 1:nrow(EGenesFactorMatrix))
colnames(EGenesFactorMatrix) <- sort(unique(unlist(pathwayEffectedGenes)))

In [ ]:
for(i in 1:length(pathwayEffectedGenes)){
    facEffectedGenes = pathwayEffectedGenes[[i]]
    EGenesFactorMatrix[i,facEffectedGenes] = 1
}

In [ ]:
options(repr.plot.width=26, repr.plot.height=6)

pheatmap(EGenesFactorMatrix, color = c("white","red"), cluster_rows = F, main = "Effected genes per factor")

In [ ]:
names(pathwayGenes) <- paste0("Factor_",1:nFactors)
# The loop above builds pathwayKOs; pathwayTargets is a rename that was
# never propagated, and this cell had never been executed.
names(pathwayKOs) <- paste0("Factor_",1:nFactors)
names(pathwayEffectedGenes) <-  paste0("Factor_",1:nFactors)


In [ ]:
factorGeneGroups <- unlist(factorGeneGroups, recursive = F)
lLen <- unlist(lapply(factorGeneGroups, length))
factorGeneGroups <- factorGeneGroups[lLen!=0]

In [ ]:
writePathwayFile(pathwayGenes=factorGeneGroups,
                 fileName="outputs/ICA_15_factors_effectedGenes.csv",
                 cNames= names(factorGeneGroups))

In [ ]:
length(factorGeneGroups)